<a href="https://colab.research.google.com/github/ShreyIND/ML/blob/main/kt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('diabetes.csv')

In [3]:
 df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [4]:
from sklearn.preprocessing import StandardScaler
x=df.iloc[:,:-1]
y=df.iloc[:,-1]


In [5]:
ss=StandardScaler()
x=ss.fit_transform(x)

In [6]:
!pip install -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.8 MB/s eta 0:00:00


In [7]:
import kerastuner as kt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,Sequential
from kerastuner.tuners import RandomSearch
from kerastuner.engine.hyperparameters import HyperParameters
from kerastuner.engine.hypermodel import HyperModel
from tensorflow.keras.layers import Dense,Dropout

/tmp/ipython-input-798634586.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [8]:
def build(hp):
    model = Sequential()

    for i in range(hp.Int('num_layers', min_value=2, max_value=20)):

        if i == 0:
            model.add(Dense(
                units=hp.Int('layer_' + str(i), min_value=32, max_value=512, step=32),
                activation=hp.Choice('activation_' + str(i), values=['relu', 'tanh', 'sigmoid']),
                input_dim=8
            ))
            model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        else:
            model.add(Dense(
                units=hp.Int('layer_' + str(i), min_value=32, max_value=512, step=32),
                activation=hp.Choice('activation_' + str(i), values=['relu', 'tanh', 'sigmoid'])
            ))
            model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

        model.add(Dropout(hp.Choice('dropout_' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5])))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('opt', values=['adam', 'rmsprop', 'sgd', 'nadam', 'adadelta']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [9]:
tuner =kt.RandomSearch(build,objective='val_accuracy',max_trials=5,directory='mydir',project_name='diabetes')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [11]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 43s


In [12]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'layer_0': 320,
 'activation_0': 'relu',
 'dropout0': 0.2,
 'dropout_0': 0.2,
 'layer_1': 512,
 'activation_1': 'relu',
 'dropout1': 0.4,
 'dropout_1': 0.4,
 'opt': 'rmsprop',
 'layer_2': 352,
 'activation_2': 'tanh',
 'dropout2': 0.4,
 'dropout_2': 0.3,
 'layer_3': 160,
 'activation_3': 'sigmoid',
 'dropout3': 0.2,
 'dropout_3': 0.1,
 'layer_4': 288,
 'activation_4': 'sigmoid',
 'dropout4': 0.1,
 'dropout_4': 0.3,
 'layer_5': 416,
 'activation_5': 'tanh',
 'dropout5': 0.6,
 'dropout_5': 0.3,
 'layer_6': 320,
 'activation_6': 'sigmoid',
 'dropout6': 0.4,
 'dropout_6': 0.4,
 'layer_7': 512,
 'activation_7': 'sigmoid',
 'dropout7': 0.8,
 'dropout_7': 0.1,
 'layer_8': 416,
 'activation_8': 'sigmoid',
 'dropout8': 0.4,
 'dropout_8': 0.5,
 'layer_9': 320,
 'activation_9': 'sigmoid',
 'dropout9': 0.1,
 'dropout_9': 0.3,
 'layer_10': 320,
 'activation_10': 'sigmoid',
 'dropout10': 0.5,
 'dropout_10': 0.1,
 'layer_11': 288,
 'activation_11': 'sigmoid',
 'dropout11': 0.2,
 'd

In [13]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 320)            │         2,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 320)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 320)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 352)            │       180,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 352)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 352)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           353 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 348,161 (1.33 MB)

 Trainable params: 348,161 (1.33 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(X_train,y_train,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.7672 - loss: 0.5390 - val_accuracy: 0.7662 - val_loss: 0.5439
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7935 - loss: 0.4283 - val_accuracy: 0.6494 - val_loss: 0.6363
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7812 - loss: 0.4569 - val_accuracy: 0.7597 - val_loss: 0.5414
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7560 - loss: 0.4923 - val_accuracy: 0.7727 - val_loss: 0.5519
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7827 - loss: 0.4800 - val_accuracy: 0.7597 - val_loss: 0.5527
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7772 - loss: 0.4375 - val_accuracy: 0.7597 - val_loss: 0.5444
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7786 - loss: 0.4499 - val_accuracy: 0.7727 - val_loss: 0.5402
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7960 - loss: 0.4347 - val_accurac